In [0]:
# ======================================================
# Import Libraries
# ======================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# ======================================================
# Load Cleaned Dataset
# ======================================================

DATA_PATH = "/Volumes/project/default/data/clean/cleaned_merged_dataset"

merged_df = spark.read.parquet(DATA_PATH)

print(f"Rows : {merged_df.count():,}")
print(f"Columns : {len(merged_df.columns)}")

display(merged_df.limit(5))

Rows : 2,128,605
Columns : 27


parent_asin,asin,user_id,review_rating,review_title,review_text,helpful_vote,verified_purchase,review_images,product_title,average_rating,rating_number,price,store,main_category,categories,features,description,details,product_images,videos,review_timestamp,year,month,day,quarter,week
B09Z2CBDDP,B09Z2DQ5VH,AFTQN4MTJHHVFM4FEXY5KT76OY4A,5.0,Nice organizer!,"We go through eggs like crazy, so I am loving this egg storage system. It saves a lot of space! I especially like that I can see exactly how many eggs we have on hand!!! The clear plastic is thick than I expected and well made. I also like that other items in the fridge can be set on top and not worry about them getting crushed. I expect this egg storage system to last us a long time and we've very pleased with it.",0,false,List(),"Egg Storage Container, Realife Automatic Rolling Egg Organizer Clear Plastic Holder for Refrigerator, 1 Layer",4.3,79,12.99,realife,Tools & Home Improvement,"List(Appliances, Parts & Accessories, Refrigerator Parts & Accessories, Egg Trays)","List([Auto Rolling Egg Storage]: The egg storage organizer for refrigerator adopts 7° slope design which makes eggs roll down automatically to the front place, easy to take out the egg from the holder without opening the lid., [ Stackable Layer Design]: Our egg container is designed with stackable layers which fixed with a card slot. Each layer can store 18 eggs keeping the refrigerator in order, no mess anymore. Semi-enclosed design can provide convenience for your life as well as good ventilation., [Transparent Egg Container ]: The automatic rolling egg organizer with lid is clearly visible adopting a transparent body which is easy to see and convenient for timely replenishment. The detachable cover is designed to load eggs conveniently. Storage grooves of the lid top can hold up to extra 9 eggs., [Food-Grade Plastic]: This egg storage container uses premium PET plastic, BPA free, durable and easy to clean. Low temperature resistant, suitable to keep in the fridge., [ Size for Wide Application ]:11""L×9""W×2.8""H one layer egg storage holder is not only perfect for refrigerator, but also for kitchen, restaurant, cabinets, table, countertop and racks.)",List(),"Map(Package Dimensions -> 11.97 x 9.57 x 3.27 inches, Pattern -> Single layer, Number of Pieces -> 1, Best Sellers Rank -> {""Tools & Home Improvement"": 157091, ""Refrigerator Egg Trays"": 216}, Size -> S, Batteries Required? -> No, Material -> Polypropylene, Plastic, Shape -> Rectangular, Manufacturer -> realife, Item Weight -> 1 pounds, Date First Available -> April 28, 2022, Room Type -> Kitchen, Handle Material -> Plastic, Brand -> Realife, Country of Origin -> China, Color -> Transparent, Batteries Included? -> No)","List(Map(thumb -> https://m.media-amazon.com/images/I/419J2mecTFL._AC_US75_.jpg, large -> https://m.media-amazon.com/images/I/419J2mecTFL._AC_.jpg, variant -> MAIN, hi_res -> https://m.media-amazon.com/images/I/71fa74TCNpL._AC_SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/41cXAzPkPhL._AC_US75_.jpg, large -> https://m.media-amazon.com/images/I/41cXAzPkPhL._AC_.jpg, variant -> PT01, hi_res -> https://m.media-amazon.com/images/I/71Qgd+EEdVL._AC_SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/41xzKnbOCGL._AC_US75_.jpg, large -> https://m.media-amazon.com/images/I/41xzKnbOCGL._AC_.jpg, variant -> PT02, hi_res -> https://m.media-amazon.com/images/I/71xWYH-S6SL._AC_SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/31286KypXzL._AC_US75_.jpg, large -> https://m.media-amazon.com/images/I/31286KypXzL._AC_.jpg, variant -> PT03, hi_res -> https://m.media-amazon.com/images/I/61KQhWPfsZL._AC_SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/41eUiwBJTSL._AC_US75_.jpg, large -> https://m.media-amazon.com/images/I/41eUiwBJTSL._AC_.jpg, variant -> PT04, hi_res -> https://m.media-amazon.com/images/I/71OSxGYZnHL._AC_SL1500_.jpg), Map(thumb -> https://m.media-amazon.com/images/I/41hatuFxCLL._AC_US75

In [0]:
# ======================================================
# Population Dataset
# ======================================================

print("Population Dataset")

print(f"Total Reviews : {merged_df.count():,}")

print(f"Total Columns : {len(merged_df.columns)}")

Population Dataset
Total Reviews : 2,128,605
Total Columns : 27


In [0]:
# ======================================================
# Year-wise Review Count
# ======================================================

year_distribution = (
    merged_df
    .groupBy("year")
    .count()
    .orderBy("year")
)

display(year_distribution)

year,count
2000,1
2002,1
2003,8
2004,16
2005,80
2006,182
2007,621
2008,883
2009,1539
2010,3537


In [0]:
from pyspark.sql import functions as F

In [0]:
# ======================================================
# Minimum Reviews Validation
# ======================================================

minimum_reviews = (
    year_distribution
    .agg(F.min("count"))
    .collect()[0][0]
)

print("Minimum Reviews in Any Year :", minimum_reviews)

if minimum_reviews >= 10000:
    print("✅ Population validation passed.")
else:
    print("❌ Some years contain fewer than 10,000 reviews.")

Minimum Reviews in Any Year : 1
❌ Some years contain fewer than 10,000 reviews.


In [0]:
# ======================================================
# Validate Years Used for Sampling
# ======================================================

sampling_population = (
    merged_df
    .filter(F.col("year").between(2014, 2023))
)

year_distribution = (
    sampling_population
    .groupBy("year")
    .count()
    .orderBy("year")
)

display(year_distribution)

year,count
2014,77353
2015,122696
2016,153916
2017,169785
2018,189599
2019,261004
2020,311251
2021,345120
2022,299487
2023,125053


In [0]:
minimum_reviews = (
    year_distribution
    .agg(F.min("count"))
    .collect()[0][0]
)

print(f"Minimum Reviews : {minimum_reviews:,}")

if minimum_reviews >= 10000:
    print("✅ Population validation passed.")
else:
    print("❌ Validation failed.")

Minimum Reviews : 77,353
✅ Population validation passed.


In [0]:
# ======================================================
# Stratified Sampling (2014–2023)
# ======================================================

from pyspark.sql.window import Window
from pyspark.sql import functions as F

window_spec = (
    Window
    .partitionBy("year")
    .orderBy(F.rand(seed=42))
)

sample_100k = (
    sampling_population
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") <= 10000)
    .drop("row_num")
)

print("✅ Stratified Sampling Completed Successfully")
print(f"Total Sample Size : {sample_100k.count():,}")

✅ Stratified Sampling Completed Successfully
Total Sample Size : 100,000


In [0]:
# ======================================================
# Sample Validation
# ======================================================

sample_distribution = (
    sample_100k
    .groupBy("year")
    .count()
    .orderBy("year")
)

display(sample_distribution)

year,count
2014,10000
2015,10000
2016,10000
2017,10000
2018,10000
2019,10000
2020,10000
2021,10000
2022,10000
2023,10000


In [0]:
# ======================================================
# Population vs Sample Comparison
# ======================================================

population = (
    sampling_population
    .groupBy("year")
    .count()
    .withColumnRenamed("count", "Population")
)

sample = (
    sample_100k
    .groupBy("year")
    .count()
    .withColumnRenamed("count", "Sample")
)

comparison = (
    population
    .join(sample, "year")
    .orderBy("year")
)

display(comparison)

year,Population,Sample
2014,77353,10000
2015,122696,10000
2016,153916,10000
2017,169785,10000
2018,189599,10000
2019,261004,10000
2020,311251,10000
2021,345120,10000
2022,299487,10000
2023,125053,10000


In [0]:
# ======================================================
# Save Sample Dataset
# ======================================================

SAMPLE_PATH = "/Volumes/project/default/data/sample100k/"
(
    sample_100k
    .write
    .mode("overwrite")
    .parquet(SAMPLE_PATH)
)

print("✅ Sample dataset saved successfully.")

✅ Sample dataset saved successfully.


In [0]:
# ======================================================
# Sampling Report
# ======================================================

report = [
    ("Population Dataset", merged_df.count()),
    ("Sampling Population (2014–2023)", sampling_population.count()),
    ("Final Sample Size", sample_100k.count()),
    ("Sampling Method", "Stratified Random Sampling"),
    ("Sampling Variable", "Review Year"),
    ("Years Covered", "2014–2023"),
    ("Rows Per Year", "10,000"),
    ("Random Seed", "42")
]

report_df = spark.createDataFrame(report, ["Metric", "Value"])

display(report_df)

Metric,Value
Population Dataset,2128605
Sampling Population (2014–2023),2055264
Final Sample Size,100000
Sampling Method,Stratified Random Sampling
Sampling Variable,Review Year
Years Covered,2014–2023
Rows Per Year,"10,000"
Random Seed,42
